In [12]:
%load_ext autoreload
%autoreload 2

import util as yu
from util import *
import util_moments as yum

yu.setpath('plot_paper')

enss=['b','c','d','e']
stouts=[5,7,10,13,15,20]
stout_chosen=10

tfphys=[0.4,0.5,0.6,0.7,0.8,0.9,1.0]
tcphys=[0.05,0.1,0.15,0.2,0.25,0.3]
tftcphys=[(tfphy,tcphy) for tfphy in tfphys for tcphy in tcphys if tfphy>=2*tcphy]

ens2Njk={'b':725,'c':400,'d':493,'e':516}

js_conn=['j+;conn','j-;conn']
js_discq=['j+;disc','js;disc','jc;disc'] 
js_gluon=[f'jg;stout{nst}' for nst in stouts]
js_disc=js_discq+js_gluon

mpl.rcParams['lines.markersize'] = mpl.rcParams['errorbar.capsize'] = 8
mpl.rcParams['axes.labelsize'] = mpl.rcParams['axes.titlesize'] = mpl.rcParams['xtick.labelsize'] = mpl.rcParams['ytick.labelsize'] = 28
# mpl.rcParams['font.size'] = 20
yu.mpl_global_elinewidth=yu.mpl_global_capthick=4

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [13]:
ens2c2pt={}; ens2moms_2pt={}; ens2c2pt0={}; ens2Njk={}
for ens in enss:
    basepath=f'/p/project1/ngff/li47/code/projectData/05_moments/{yu.ens2full[ens]}/data_merge/'
    if ens in ['e']:
        path=f'{basepath}disc_2pt_N=100,127,84_sup.h5'
        with h5py.File(path) as f:
            moms_2pt=yu.moms2list(f['moms'])
            c2pt=yu.jackknife(np.real(f['data/N_N'][:,:,:]))
            c2pt=yu.superjackknife(c2pt,yum.key2cfgs[(ens,'N')],yum.key2cfgs[(ens,'all')])
    else:
        path=f'{basepath}disc_2pt.h5'
        with h5py.File(path) as f:
            moms_2pt=yu.moms2list(f['moms'])
            c2pt=yu.jackknife(np.real(f['data/N_N'][:,:,:]))
                
    ens2moms_2pt[ens]=moms_2pt
    ens2c2pt[ens]=c2pt
    ens2c2pt0[ens]=c2pt[:,:,moms_2pt.index([0,0,0])]
    ens2Njk[ens]=len(c2pt)
    print(ens,len(c2pt))

b 725
c 400
d 493
e 516


In [14]:
# global parameters
ens2tminss={
        'b':[range(8,25+1),range(1,10+1),range(1,4+1)],
        'c':[range(8,29+1),range(1,16+1),range(1,5+1)],
        'd':[range(8,33+1),range(1,18+1),range(1,6+1)],
        'e':[range(8,39+1),range(1,18+1),range(1,5+1)],
    }
ens2tminss_large={
        'b':[range(8,25+1),range(1,9+1),range(1,1+1)],
        'c':[range(8,29+1),range(1,10+1),range(1,1+1)],
        'd':[range(8,33+1),range(1,11+1),range(1,1+1)],
        'e':[range(8,39+1),range(1,13+1),range(1,1+1)],
    } # used for large momenta

# ens2selections={
#     'b':{'1st':20,'2st':7,'3st':3},
#     'c':{'1st':21,'2st':8,'3st':3},
#     'd':{'1st':24,'2st':9,'3st':3},
#     'e':{'1st':32,'2st':11,'3st':4},
# }

ens2selections={}
tfphy_1st=1.6; tfphy_2st=0.6; tfphy_3st=0.2
for ens in enss:
    ens2selections[ens]={}
    ens2selections[ens]['1st']=round(tfphy_1st/yu.ens2a[ens])
    ens2selections[ens]['2st']=round(tfphy_2st/yu.ens2a[ens])
    ens2selections[ens]['3st']=round(tfphy_3st/yu.ens2a[ens])

ens2selections

{'b': {'1st': 20, '2st': 8, '3st': 3},
 'c': {'1st': 23, '2st': 9, '3st': 3},
 'd': {'1st': 28, '2st': 11, '3st': 4},
 'e': {'1st': 33, '2st': 12, '3st': 4}}

In [15]:
fontsize=22
ens2pars_jk_meff1st={}; ens2pars_jk_meff2st={}; ens2pars_jk_meff3st={}
for ens in enss[:]:
    meff=yu.jackmap(yu.c2pt2meff,ens2c2pt0[ens])
    tminss=ens2tminss[ens]

    # tmins=[1.6,0.6,0.2]
    # tmins=[1.6,0.6,0.2]
    # tmins=[t*yu.ens2a['b'] for t in [20,7,3]]
    # selections={f'{i+1}st':yu.find_t_cloest(tmins[i],yu.ens2a[ens]) for i in range(3)}
    selections=ens2selections[ens]
    # selections={}
    print(ens,selections)
    
    fitss_2pt=yu.getFits(f'meff_{ens}',pathlabel='analysis_c2pt')
    # yu.doFits_meff_nst(meff,tminss,[0.4,0.5,2,0.8,1],downSampling=1,label=f'meff_{ens}',overwrite=overwrite)
    fig,axd,result=yu.makePlot_2pt_SimoneStyle(meff,fitss_2pt,xunit=yu.ens2a[ens],yunit=yu.ens2aInv[ens]/1000,ylims='std_N',\
        selection=selections,chi2Q=False,bandQ=False,legendQ=False)
    fig.set_size_inches(11,8)
    gs = axd['f1'].get_subplotspec().get_gridspec()
    gs.set_width_ratios([1, 1, 1.4])
    fig.canvas.draw_idle()

    fig.suptitle(yu.ens2label[ens],fontsize=32,y=0.92)
    
    axd['f1'].set_xlim([0.3,2.3])
    axd['f2'].set_xlim([0,2.1])
    axd['f3'].set_xlim([0,1.2])
    axd['f3'].set_ylim([0,5])
    axd['f2'].set_xticks([0,0.5,1.0,1.5,2.0])
    axd['f3'].set_xticks([0,0.5,1.0])
    axd['f3'].set_yticks([1,2,3,4])
    axd['f1'].set_ylabel(r'$m_N^{\rm eff}$ [GeV]')
    axd['f2'].set_ylabel(r'$m_N$ [GeV]')
    axd['f3'].set_ylabel(r'$E_1(\vec{0})$ [GeV]')
    
    m,e=yu.jackme(result['2st'][:,0]*yu.ens2aInv[ens]/1000)
    axd['f1'].axhspan(m-e,m+e,color='g',alpha=0.2)
    axd['f2'].axhspan(m-e,m+e,color='g',alpha=0.2)
    m,e=yu.jackme((result['2st'][:,0]+result['2st'][:,1])*yu.ens2aInv[ens]/1000)
    axd['f3'].axhspan(m-e,m+e,color='g',alpha=0.2)

    handles=[plt.errorbar(np.nan,np.nan,yerr=np.nan,color=color,fmt=fmt) for color,fmt in zip(yu.colors8[:3],yu.fmts8[:3])]
    
    yu.legend(axd['f2'],labels=[rf'$m_N^{{\rm {n}st}}$' for n in [1,2,3]],ncols=3,tightQ=True,handles=handles,fontsize=fontsize)
    yu.legend(axd['f3'],labels=[rf'$E_1^{{\rm {n}st}}(\vec{{0}})$' for n in [2,3]],ncols=1,tightQ=True,handles=handles[1:],fontsize=fontsize)
    
    yu.finalizePlot(f'c2pt/meff_{ens}',closeQ=True,mkdirQ=True)
    
    ens2pars_jk_meff1st[ens]=result['1st']
    ens2pars_jk_meff2st[ens]=result['2st']
    ens2pars_jk_meff3st[ens]=result['3st']

b {'1st': 20, '2st': 8, '3st': 3}
c {'1st': 23, '2st': 9, '3st': 3}
d {'1st': 28, '2st': 11, '3st': 4}
e {'1st': 33, '2st': 12, '3st': 4}


In [16]:
ens2sgm=yu.load_pkl('data_aux/dat_ignore/ens2sgm.pkl')
for ens in enss:
    m,e=yu.jackme(ens2sgm[ens])
    ens2sgm[ens]=yu.jackknife_pseudo(m,e,ens2Njk[ens])[:,0]
ens2DmN_sgm={ens:ens2sgm[ens]*(yu.ens2amul_iso[ens]-yu.ens2amul[ens])/yu.ens2amul[ens] for ens in enss}
print({ens:yu.jackme_un2str(ens2sgm[ens]) for ens in enss},'sgm')
print({ens:yu.jackme_un2str(ens2DmN_sgm[ens]) for ens in enss},'DmN_sgm')

path='data_aux/DmN_ChPT.csv'
df=pd.read_csv(path, index_col=0)
def func(ele):
    m,e=ele.split(',')
    m=m.split('(')[-1]
    e=e.split(')')[0]
    return float(m),float(e)
dic_DmN_ChPT={(row_label, col_label): func(df.loc[row_label, col_label])
    for row_label in df.index
    for col_label in df.columns
}
ens2DmN_N2LO={ens:dic_DmN_ChPT[(yu.ens2label[ens],'DmN_N2LO')] for ens in enss}
ens2DmN_N2LO={ens:yu.jackknife_pseudo(ens2DmN_N2LO[ens][0],ens2DmN_N2LO[ens][1],ens2Njk[ens])[:,0] for ens in enss}
print({ens:yu.jackme_un2str(ens2DmN_N2LO[ens]) for ens in enss},'DmN_N2LO')

fig,axs=yu.getFigAxs(3,2,Lrow=3,Lcol=5,gridspec_kw={'hspace': 0.1, 'wspace': 0.05},sharex=True,sharey=True)
yu.addColHeader(axs,[r'LAT',r'$\chi PT$'],y=1.05,fontsize=24)

for inst,nst in enumerate(['1st','2st','3st']):
    ax=axs[inst,0]
    ax.set_ylabel(rf'$m_N^{{{nst}}}$ [GeV]')
    ax.set_ylim([0.91,0.97])
    ax.set_yticks([0.92,0.94,0.96])

dic={}
for iDmN,DmN in enumerate(['sgm','N2LO']):
    for inst,nst in enumerate(['1st','2st','3st']):
        ax=axs[inst,iDmN]
        
        ens2dat={ens:(globals()[f'ens2pars_jk_meff{nst}'][ens][:,0]*yu.ens2aInv[ens] + globals()[f'ens2DmN_{DmN}'][ens])/1000  for ens in enss}
        fits=yu.doFits_continuumExtrapolation(ens2dat,lat_a2s_plt=yum.lat_a2s_plt)
        pars_jk,probs_jk=yu.jackMA(fits)
        
        x=yum.lat_a2s_plt
        y,e=yu.jackme(pars_jk)
        c='r'
        ax.plot(x, y, "--", color=c, lw=2)
        ax.fill_between(x, y - e, y + e, color=c, alpha=0.12)
        dic[(DmN,nst)]=yu.jackme_un2str(pars_jk[:,0]*1000)
        # print(yu.jackme_un2str(pars_jk[:,0]-((yu.m_proton+yu.m_neutron)/2/1000)))
        
        for ens in enss:
            mean,err=yu.jackme(ens2dat[ens])
            plt_x=yu.ens2a[ens]**2; plt_y,plt_yerr=mean,err
            ax.errorbar(plt_x,plt_y,plt_yerr,color='r')
            
        if inst==2:
            ax.set(xlabel=r"$a^2\ [\mathrm{fm}^2]$", xlim=(0, 0.008), xticks=[0.000, 0.003, 0.006], xticklabels=[0, 0.003, 0.006])

yu.finalizePlot('c2pt/mN_ce',tightQ=False)

{'b': '50.3(3.1)', 'c': '46.3(2.7)', 'd': '52.7(5.1)', 'e': '42.8(4.0)'} sgm
{'b': '-3.71(23)', 'c': '-1.049(61)', 'd': '-4.55(44)', 'e': '-0.914(86)'} DmN_sgm
{'b': '-4.50(38)', 'c': '-1.32(47)', 'd': '-5.33(45)', 'e': '-1.24(44)'} DmN_N2LO


In [17]:
t1=[[yu.jackme_un2str(globals()[f'ens2pars_jk_meff{nst}'][ens][:,0]*yu.ens2aInv[ens]) for nst in ['1st','2st','3st']]+[yu.jackme_un2str(ens2DmN_sgm[ens],precision=1),yu.jackme_un2str(ens2DmN_N2LO[ens],precision=1)] for ens in enss]
t2=[[dic[(DmN,nst)] for nst in ['1st','2st','3st']] for DmN in ['sgm','N2LO']]

df = pd.DataFrame('', index=[yu.ens2label[ens] for ens in enss] + ['LAT',r'$\chi$PT'], columns=[rf'$m_N^{{{nst}}}$' for nst in ['1st','2st','3st']]+[r'$\Delta m_N^{\rm LAT}$',r'$\Delta m_N^{\rm \chi PT}$'])
df.iloc[:len(t1)]=t1
df.iloc[-2:,:3]=t2

t=df.loc['LAT',r'$m_N^{2st}$']
df.loc['LAT',r'$m_N^{2st}$']=r'\setlength{\fboxsep}{1pt}\colorbox{gray!20}{'+t+r'}'

tex=df.to_latex(
    escape=False,          # allow latex in labels
    column_format='cccccc',
    bold_rows=False,
)
tex = tex.replace(r'\toprule'+'\n',r'')
tex = tex.replace(r'\midrule', r'\hline')
tex = tex.replace(r'\bottomrule'+'\n',r'')
tex = tex.replace('\nLAT &',' \\hline\nLAT &')
print(tex)

\begin{tabular}{cccccc}
 & $m_N^{1st}$ & $m_N^{2st}$ & $m_N^{3st}$ & $\Delta m_N^{\rm LAT}$ & $\Delta m_N^{\rm \chi PT}$ \\
\hline
B64 & 943.5(6.2) & 941.8(4.6) & 941.4(6.2) & -3.7(2) & -4.5(4) \\
C80 & 942.4(5.8) & 941.2(4.5) & 940.4(5.0) & -1.05(6) & -1.3(5) \\
D96 & 949.7(7.7) & 946.4(4.2) & 946.0(4.1) & -4.5(4) & -5.3(4) \\
E112 & 941.3(9.9) & 941.5(4.3) & 941.6(4.4) & -0.91(9) & -1.2(4) \\ \hline
LAT & 942.6(4.7) & \setlength{\fboxsep}{1pt}\colorbox{gray!20}{941.2(2.9)} & 941.1(3.1) &  &  \\
$\chi$PT & 942.2(4.8) & 940.8(2.9) & 940.7(3.1) &  &  \\
\end{tabular}



In [18]:
ens2msq2pars_jk=yu.load_pkl_reg('ens2msq2pars_jk',pathlabel='analysis_c2pt')

fig,axs=yu.getFigAxs(2,1,Lrow=3,Lcol=10,sharex=True)
ax=axs[0,0]
ax.set_ylim([0.4,3.4])
ax.set_ylabel(r'$E_{n}(\vec{p})$ [GeV]')
ax=axs[1,0]
ax.set_ylim([0.5,1.5])
ax.set_yticks([0.7,1.0,1.3])
ax.set_ylabel(r'$r_1(\vec{p})$')

ax=axs[1,0]
ax.set_xlim([-0.02,1.4])
ax.set_xlabel(r'$|\vec{p}|^2$ [GeV$^2$]')

for iens,ens in enumerate(enss):
    xunit=(2*np.pi/yu.ens2NL[ens]*yu.ens2aInv[ens]/1000)**2; yunit=yu.ens2aInv[ens]/1000
    msq2pars_jk=ens2msq2pars_jk[ens]
    for msq in msq2pars_jk.keys():
        pars_jk=msq2pars_jk[msq]
        
        ax=axs[0,0]
        m,e=yu.jackme(pars_jk[:,0])
        plt_x=(msq+0.1*iens)*xunit; plt_y=m*yunit; plt_yerr=e*yunit
        ax.errorbar(plt_x,plt_y,plt_yerr,color=yu.colors8[iens],fmt=yu.fmts8[iens],mfc='white',label=r'$E_N(\vec{p})$' if iens==0 and msq==1 else None)
        
        m,e=yu.jackme(pars_jk[:,0]+pars_jk[:,1])
        plt_x=(msq+0.1*iens)*xunit; plt_y=m*yunit; plt_yerr=e*yunit
        ax.errorbar(plt_x,plt_y,plt_yerr,color=yu.colors8[iens],fmt=yu.fmts8[iens],label=yu.ens2label[ens] if msq==0 else r'$E_1(\vec{p})$'  if iens==0 and msq==1 else None)
        
        ax=axs[1,0]
        m,e=yu.jackme(pars_jk[:,2])
        plt_x=(msq+0.1*iens)*xunit; plt_y=m*1; plt_yerr=e*1
        ax.errorbar(plt_x,plt_y,plt_yerr,color=yu.colors8[iens],fmt=yu.fmts8[iens])

ax=axs[0,0]

yu.legend(ax,ncols=4,tightQ=True,order=[0,2,3,1,4,5],fontsize=20)
yu.finalizePlot('c2pt/pars_2st',closeQ=True)